In [2]:
import pickle
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from copy import deepcopy

from utils import pareto_frontier, hex2rgba

In [71]:
data = 'temporal'
base_model = 'nn'
cost_fn = 'L1'
with open(f'../results/tradeoff_{base_model}_{data}.pkl', 'rb') as f:
    df_results = pickle.load(f)
    if base_model == 'lr':
        df_results.rename(columns={'id': 'alg'}, inplace=True)

In [72]:
df_results_avg = df_results.groupby(['alg', 'beta', 'p'], as_index=False).mean()
df_results_avg[['robustness_std', 'consistency_std']] = df_results.groupby(['alg', 'beta', 'p'], as_index=False).std(ddof=0, numeric_only=True)[['robustness', 'consistency']].clip(0.)
df_results_avg = df_results_avg.sort_values(['alg', 'p', 'beta'])

In [73]:
df_im = deepcopy(df_results_avg)
df_im['alg'] = df_im['alg'].apply(lambda x: 'OPT' if x not in ['OPT', 'ROAR'] else x)

In [74]:
colors = px.colors.qualitative.Plotly
nc = len(colors)
band_colors = [hex2rgba(c, 0.2) for c in colors]
font_family = 'Times New Roman'
font_color = 'black'
width, height = 720, 540
symbols = ['circle', 'square', 'triangle-up', 'cross', 'x']
show_errors = True

fig = go.Figure()
for p in df_im['p'].unique():
    for ii, alg in enumerate(df_im['alg'].unique()):
        temp = df_im[(df_im['p'] == p) & (df_im['alg'] == alg)].sort_values(['consistency'], ascending=False)
        mask = pareto_frontier(temp['robustness'], temp['consistency'])
        temp = temp.iloc[mask]
        t = '\\theta_{p}'.format(p=f'{p}')
        if alg in df_im['alg'].unique():
            if alg == 'ROAR':
                name = f'({temp["consistency"].round(2).iloc[0]:.2f}, {temp["robustness"].round(2).iloc[0]:.2f})'
            else:
                if base_model == 'lr':
                    name = r'$\hat{theta}$'.format(theta=t) if p != 0 else r'$\theta_{0}$'
                else:
                    name = r'$\hat{theta}$'.format(theta=t)

            fig.add_trace(go.Scatter(
                x = temp['consistency'] if alg != 'ROAR' else [None],
                y = temp['robustness'] if alg != 'ROAR' else [None],
                # x = temp['consistency'],
                # y = temp['robustness'],
                marker = dict(color=colors[p] if alg in ['OPT', 'ROAR'] else colors[(p+ii)%nc], symbol=symbols[p] if alg != 'ROAR' else 'star', size=2 if alg != 'ROAR' else 10),
                mode = 'markers+lines' if alg != 'ROAR' else 'markers',
                name = name,
                showlegend=True,
                customdata=temp['beta'],
                hovertemplate='consistency: %{x}<br>robustness: %{y}<br>beta: %{customdata}'
            ))

            if show_errors:
                if alg != 'ROAR':

                    fig.add_trace(go.Scatter(
                        x = temp['consistency'] + temp['consistency_std'],
                        y=temp['robustness'],
                        marker = dict(color=colors[p]),
                        mode='lines',
                        line=dict(width=0),
                        showlegend=False
                    ))
                    
                    fig.add_trace(go.Scatter(
                        x = (temp['consistency'] - temp['consistency_std']).clip(0),
                        y=temp['robustness'],
                        marker = dict(color=colors[p]),
                        mode='lines',
                        line=dict(width=0),
                        fillcolor=band_colors[p],
                        fill='tonexty',
                        showlegend=False
                    ))

fig.update_xaxes(
    title=dict(
        text='Consistency',
        font=dict(
            family=font_family,
            color=font_color,
            size=25
        )
        ), 
    showline=True, 
    mirror=True,
    linecolor='black', 
    gridcolor='lightgrey', 
    zerolinewidth=1,
    zerolinecolor='lightgrey',
    )


fig.update_yaxes(
    title=dict(
        text='Robustness',
        font=dict(
            family=font_family,
            color=font_color,
            size=25
        ), 
        ), 
    showline=True, 
    mirror=True,
    linecolor='black', 
    gridcolor='lightgrey',
    zerolinewidth=1,
    zerolinecolor='lightgrey',
    )


fig.update_layout(
    legend=dict(
        x=0.975, 
        y=0.975, 
        xanchor='right',
        font=dict(
            family=font_family,
            color=font_color,
            size=15
            ), 
        bgcolor='rgba(255, 255, 255, 0.7)',
        bordercolor='lightgrey',
        borderwidth=1,
        entrywidth=0.1,
        entrywidthmode='pixels',
        ),
    width=width,
    height=height,
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        tickfont=dict(
            family=font_family,
            color=font_color,
            size=20,
        )
    ),
    yaxis=dict(
        tickfont=dict(
            family=font_family,
            color=font_color,
            size=20
        )
    )
    )

fig.show()

In [75]:
fig.write_image(f'../figs/tradeoff_{base_model}_{data}_con_err.pdf')